# Classification

### Import Libraries and seed
Import the necessary libraries for data processing, model building, training, and evaluation. Adding a seed ensures reproducibility by making sure that the random number generation is consistent across different runs.

In [1]:
import os
import re
import random
import time
import json
import sys as _sys
import hashlib as _hl

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from collections import Counter

from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import confusion_matrix
from sklearn.inspection import permutation_importance
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

from sklearn.naive_bayes import GaussianNB
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score as _acc, precision_score as _prec, recall_score as _rec, f1_score as _f1
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis

from scipy.signal import savgol_filter

def set_seed(seed):
    np.random.seed(seed)
    random.seed(seed)
    return seed

# Set seed for reproducibility
seed = set_seed(42)

# Get the current directory of the notebook
notebook_dir = os.getcwd()

## Prepare Training Data
### Split data into X and y


In [2]:
def preprocess_data(df, labels, freqs, eliminate_std_dev=False, eliminate_LG=False, drop_sample=True):
    # Reduce number of different samples for testing
    X_ = df[df['Sample'].isin(labels)]

    y_ = X_['Sample']

    if drop_sample:
        X_ = X_.drop(columns=['Sample'])

    if freqs:
        # Subset of specific frequencies to use as input features (or without mean)
        columns = [f'{freq}.0 HG (mV) mean' for freq in freqs] + \
                  [f'{freq}.0 LG (mV) mean' for freq in freqs] + \
                  [f'{freq}.0 HG (mV)' for freq in freqs] + \
                  [f'{freq}.0 LG (mV)' for freq in freqs] + \
                  [f'{freq}.0 HG (mV) std deviation' for freq in freqs] + \
                  [f'{freq}.0 LG (mV) std deviation' for freq in freqs] + \
                  ['Sample']


        # Filter columns that exist in X_
        existing_columns = [col for col in columns if col in X_.columns]

        # Check if existing_columns is empty
        if not existing_columns:
            print("No matching columns found in X_.")
        else:
            X_ = X_[existing_columns]

        # Sort columns by frequency value
        X_ = X_.reindex(sorted(X_.columns), axis=1)

    if eliminate_std_dev:
        # Eliminate std dev columns from the input features
        X_ = X_.drop(columns=[col for col in X_.columns if 'std deviation' in col])

    if eliminate_LG:
        # Eliminate LG columns from the input features
        X_ = X_.drop(columns=[col for col in X_.columns if 'LG' in col])

    return X_, y_


### Define Models
- Random Forest
- Naive-Bayes
- Logistic Regression
- Gradient Boosting
- Support Vector Machine

In [3]:
def _lr_coef(lr_model):
    """Return LR coef_ matrix whether lr_model is a bare estimator or a Pipeline."""
    if hasattr(lr_model, 'named_steps'):
        for _step in ('logisticregression', 'clf', 'lr'):
            if _step in lr_model.named_steps and hasattr(lr_model.named_steps[_step], 'coef_'):
                return lr_model.named_steps[_step].coef_
    return lr_model.coef_


### Train all Models

In [4]:
def train_models(X_train, y_train, seed):
    training_times = []

    # Random Forest
    start_time = time.time()
    rf_model = RandomForestClassifier(random_state=seed)
    rf_model.fit(X_train, y_train)
    training_times.append(time.time() - start_time)

    # Naive Bayes
    start_time = time.time()
    nb_model = GaussianNB()
    nb_model.fit(X_train, y_train)
    training_times.append(time.time() - start_time)

    start_time = time.time()
    lr_model = make_pipeline(StandardScaler(),
                             LogisticRegression(random_state=seed, max_iter=5000))
    lr_model.fit(X_train, y_train)
    training_times.append(time.time() - start_time)

    # Gradient Boosting
    start_time = time.time()
    gb_model = GradientBoostingClassifier(random_state=seed)
    gb_model.fit(X_train, y_train)
    training_times.append(time.time() - start_time)

    # SVM
    start_time = time.time()
    svm_model = SVC(random_state=seed)
    svm_model.fit(X_train, y_train)
    training_times.append(time.time() - start_time)

    return rf_model, nb_model, lr_model, gb_model, svm_model, training_times


In [5]:
def get_feature_importances(rf_model, lr_model, gb_model, nb_model, svm_model, X_train, y_train, seed, plot=True, n=10):
    feature_names = X_train.columns

    # Random Forest feature importances
    rf_feature_importances = rf_model.feature_importances_
    rf_feature_importances_df = pd.DataFrame({'Feature': feature_names, 'Importance': rf_feature_importances})
    rf_feature_importances_df = rf_feature_importances_df.sort_values('Importance', ascending=False)

    # Logistic Regression feature importances (pipeline-aware: see _lr_coef)
    lr_feature_importances = _lr_coef(lr_model)[0]
    lr_feature_importances_df = pd.DataFrame({'Feature': feature_names, 'Importance': lr_feature_importances})
    lr_feature_importances_df = lr_feature_importances_df.sort_values('Importance', ascending=False)

    # Gradient Boosting feature importances
    gb_feature_importances = gb_model.feature_importances_
    gb_feature_importances_df = pd.DataFrame({'Feature': feature_names, 'Importance': gb_feature_importances})
    gb_feature_importances_df = gb_feature_importances_df.sort_values('Importance', ascending=False)

    # Naive Bayes permutation importance
    result_nb = permutation_importance(nb_model, X_train, y_train, n_repeats=5, random_state=seed, n_jobs=1)
    sorted_idx_nb = result_nb.importances_mean.argsort()[::-1]
    nb_feature_importances_df = pd.DataFrame({'Feature': feature_names[sorted_idx_nb], 'Importance': result_nb.importances_mean[sorted_idx_nb]})

    # SVM permutation importance
    result_svm = permutation_importance(svm_model, X_train, y_train, n_repeats=5, random_state=seed, n_jobs=1)
    sorted_idx_svm = result_svm.importances_mean.argsort()[::-1]
    svm_feature_importances_df = pd.DataFrame({'Feature': feature_names[sorted_idx_svm], 'Importance': result_svm.importances_mean[sorted_idx_svm]})

    if plot:
        # Set standard font family
        plt.rcParams['font.family'] = 'Arial'  # or 'Arial', 'Times New Roman', etc.

        # Create directory for saving feature importance plots
        feature_imp_path = os.path.normpath(os.path.join(notebook_dir, '..', '..', 'data/results/feature_importance_detailed/'))
        if not os.path.exists(feature_imp_path):
            os.makedirs(feature_imp_path)

        # Define enhanced color schemes for each model
        colors = {
            'RF': plt.cm.viridis(np.linspace(0.2, 0.8, n)),
            'LR': plt.cm.plasma(np.linspace(0.2, 0.8, n)),
            'GB': plt.cm.inferno(np.linspace(0.2, 0.8, n)),
            'NB': plt.cm.cividis(np.linspace(0.2, 0.8, n)),
            'SVM': plt.cm.magma(np.linspace(0.2, 0.8, n))
        }

        # Random Forest Plot
        fig, ax = plt.subplots(figsize=(20, 10))
        bars = ax.barh(rf_feature_importances_df['Feature'][:n],
                      rf_feature_importances_df['Importance'][:n],
                      color=colors['RF'],
                      edgecolor='white',
                      linewidth=0.8,
                      alpha=0.85)

        # Add gradient effect to bars
        for i, bar in enumerate(bars):
            bar.set_facecolor(colors['RF'][i])

        ax.set_xlabel('Importance', fontsize=20, color='#2E2E2E', family='DejaVu Sans')
        ax.set_title('Random Forest Feature Importances', fontsize=22,
                    color='#2E2E2E', pad=20, family='DejaVu Sans')
        ax.tick_params(axis='x', labelsize=18, colors='#2E2E2E')
        ax.tick_params(axis='y', labelsize=18, colors='#2E2E2E')

        # Enhanced grid styling
        ax.grid(True, alpha=0.3, linestyle='-', linewidth=0.8, color='gray')
        ax.set_axisbelow(True)

        # Subtle background gradient
        ax.patch.set_facecolor('#FAFAFA')

        plt.tight_layout()
        plt.savefig(os.path.join(feature_imp_path, 'RF_detailed_feature_importance.pdf'),
                   format='pdf', bbox_inches='tight', dpi=300, facecolor='white')
        plt.show()

        # Logistic Regression Plot
        fig, ax = plt.subplots(figsize=(20, 10))
        bars = ax.barh(lr_feature_importances_df['Feature'][:n],
                      lr_feature_importances_df['Importance'][:n],
                      color=colors['LR'],
                      edgecolor='white',
                      linewidth=0.8,
                      alpha=0.85)

        for i, bar in enumerate(bars):
            bar.set_facecolor(colors['LR'][i])

        ax.set_xlabel('Importance', fontsize=18, color='#2E2E2E', family='DejaVu Sans')
        ax.set_title('Logistic Regression Feature Importances', fontsize=22,
                    color='#2E2E2E', pad=20, family='DejaVu Sans')
        ax.tick_params(axis='x', labelsize=16, colors='#2E2E2E')
        ax.tick_params(axis='y', labelsize=16, colors='#2E2E2E')
        ax.grid(True, alpha=0.3, linestyle='-', linewidth=0.8, color='gray')
        ax.set_axisbelow(True)
        ax.patch.set_facecolor('#FAFAFA')

        plt.tight_layout()
        plt.savefig(os.path.join(feature_imp_path, 'LR_detailed_feature_importance.pdf'),
                   format='pdf', bbox_inches='tight', dpi=300, facecolor='white')
        plt.show()

        # Gradient Boosting Plot
        fig, ax = plt.subplots(figsize=(20, 10))
        bars = ax.barh(gb_feature_importances_df['Feature'][:n],
                      gb_feature_importances_df['Importance'][:n],
                      color=colors['GB'],
                      edgecolor='white',
                      linewidth=0.8,
                      alpha=0.85)

        for i, bar in enumerate(bars):
            bar.set_facecolor(colors['GB'][i])

        ax.set_xlabel('Importance', fontsize=18, color='#2E2E2E', family='DejaVu Sans')
        ax.set_title('Gradient Boosting Feature Importances', fontsize=22,
                    color='#2E2E2E', pad=20, family='DejaVu Sans')
        ax.tick_params(axis='x', labelsize=16, colors='#2E2E2E')
        ax.tick_params(axis='y', labelsize=16, colors='#2E2E2E')
        ax.grid(True, alpha=0.3, linestyle='-', linewidth=0.8, color='gray')
        ax.set_axisbelow(True)
        ax.patch.set_facecolor('#FAFAFA')

        plt.tight_layout()
        plt.savefig(os.path.join(feature_imp_path, 'GB_detailed_feature_importance.pdf'),
                   format='pdf', bbox_inches='tight', dpi=300, facecolor='white')
        plt.show()

        # Naive Bayes Plot (Enhanced Boxplot)
        fig, ax = plt.subplots(figsize=(12, 8))
        bp = ax.boxplot(result_nb.importances[sorted_idx_nb][:n].T,
                       vert=False,
                       labels=X_train.columns[sorted_idx_nb][:n],
                       patch_artist=True,
                       boxprops=dict(facecolor='#8E44AD', alpha=0.8, linewidth=1.5),
                       whiskerprops=dict(color='#2E2E2E', linewidth=2),
                       capprops=dict(color='#2E2E2E', linewidth=2),
                       medianprops=dict(color='white', linewidth=3),
                       flierprops=dict(marker='o', markerfacecolor='#E74C3C', markersize=8, alpha=0.8, markeredgecolor='white'))

        ax.set_xlabel('Permutation Importance', fontsize=18, color='#2E2E2E', family='DejaVu Sans')
        ax.set_title('Naive Bayes Permutation Feature Importance', fontsize=22,
                    color='#2E2E2E', pad=20, family='DejaVu Sans')
        ax.tick_params(axis='x', labelsize=16, colors='#2E2E2E')
        ax.tick_params(axis='y', labelsize=16, colors='#2E2E2E')
        ax.grid(True, alpha=0.3, linestyle='-', linewidth=0.8, color='gray')
        ax.set_axisbelow(True)
        ax.patch.set_facecolor('#FAFAFA')

        plt.tight_layout()
        plt.savefig(os.path.join(feature_imp_path, 'NB_detailed_feature_importance.pdf'),
                   format='pdf', bbox_inches='tight', dpi=300, facecolor='white')
        plt.show()

        # SVM Plot (Enhanced Boxplot)
        fig, ax = plt.subplots(figsize=(12, 8))
        bp = ax.boxplot(result_svm.importances[sorted_idx_svm][:n].T,
                       vert=False,
                       labels=X_train.columns[sorted_idx_svm][:n],
                       patch_artist=True,
                       boxprops=dict(facecolor='#E67E22', alpha=0.8, linewidth=1.5),
                       whiskerprops=dict(color='#2E2E2E', linewidth=2),
                       capprops=dict(color='#2E2E2E', linewidth=2),
                       medianprops=dict(color='white', linewidth=3),
                       flierprops=dict(marker='o', markerfacecolor='#E74C3C', markersize=8, alpha=0.8, markeredgecolor='white'))

        ax.set_xlabel('Permutation Importance', fontsize=18, color='#2E2E2E', family='DejaVu Sans')
        ax.set_title('SVM Permutation Feature Importance', fontsize=22,
                    color='#2E2E2E', pad=20, family='DejaVu Sans')
        ax.tick_params(axis='x', labelsize=16, colors='#2E2E2E')
        ax.tick_params(axis='y', labelsize=16, colors='#2E2E2E')
        ax.grid(True, alpha=0.3, linestyle='-', linewidth=0.8, color='gray')
        ax.set_axisbelow(True)
        ax.patch.set_facecolor('#FAFAFA')

        plt.tight_layout()
        plt.savefig(os.path.join(feature_imp_path, 'SVM_detailed_feature_importance.pdf'),
                   format='pdf', bbox_inches='tight', dpi=300, facecolor='white')
        plt.show()

    return rf_feature_importances_df, lr_feature_importances_df, gb_feature_importances_df, nb_feature_importances_df, svm_feature_importances_df

### Confusion Matrix

In [6]:
def plot_confusion_matrix_pdf(y_true, y_pred, labels, save_path=None, model_name=None):
    conf_matrix = confusion_matrix(y_true, y_pred, labels=labels)

    fig, ax = plt.subplots(figsize=(8, 8))
    cmap=plt.cm.Blues
    cax = ax.matshow(conf_matrix, cmap=cmap)
    fig.colorbar(cax)

    # Determine text color based on cell value for better visibility
    for i in range(len(labels)):
        for j in range(len(labels)):
            # Calculate percentage
            percentage = conf_matrix[i, j] / np.sum(conf_matrix, axis=1)[i] * 100 if np.sum(conf_matrix, axis=1)[i] > 0 else 0

            # Determine text color based on cell darkness
            cell_value = conf_matrix[i, j]
            if cell_value > conf_matrix.max() / 3:
                text_color = 'white'

                plt.text(j, i, f'{conf_matrix[i, j]}\n{percentage:.1f}%',
                     horizontalalignment="center",
                     verticalalignment="center",
                     fontsize=8,
                     ha='center', va='center',
                     color=text_color)
            else:
                text_color = cmap(1.0)

                if conf_matrix[i, j] != 0:

                    plt.text(j, i, f'{conf_matrix[i, j]}\n{percentage:.1f}%',
                        horizontalalignment="center",
                        verticalalignment="center",
                        fontsize=8,
                        ha='center', va='center',
                        color=text_color)



    plt.xlabel('Predicted', fontweight='bold', fontsize=12)
    plt.ylabel('True', fontweight='bold', fontsize=12)
    plt.xticks(np.arange(len(labels)), labels, rotation=45, fontweight='bold')
    plt.yticks(np.arange(len(labels)), labels, fontweight='bold')
    plt.title('Confusion Matrix', fontweight='bold', fontsize=14)

    # Adjust layout to make room for rotated x labels
    plt.tight_layout()

    # Save the plot if a path is provided
    if save_path:
        # Create directory if it doesn't exist
        if not os.path.exists(save_path):
            os.makedirs(save_path)

        # Create filename
        model_suffix = f"_{model_name}" if model_name else ""
        filename = f"confusion_matrix{model_suffix}.pdf"
        filepath = os.path.join(save_path, filename)

        # Save as PDF
        plt.savefig(filepath, format='pdf', bbox_inches='tight', dpi=300)
        print(f"Confusion matrix saved to: {filepath}")
    else:
        print("Confusion matrix plot not saved.")

    plt.show()


In [7]:
def add_features(X, y, subset_freqs, HG_diff=True, LG_diff=True):

    X['Sample'] = y

    # Initialize a dictionary to store results
    mean_std_dict = {}

    for freq in subset_freqs:
        # Calculate HG and LG mean values for each frequency
        agg_dict = {}
        if f'{freq}.0 LG (mV) mean' in X.columns:
            agg_dict['LG_mean'] = (f'{freq}.0 LG (mV) mean', 'mean')
        if f'{freq}.0 HG (mV) mean' in X.columns:
            agg_dict['HG_mean'] = (f'{freq}.0 HG (mV) mean', 'mean')

        mean_std_dict[freq] = X.groupby('Sample').agg(**agg_dict).reset_index()

        mean_std_dict[freq]['Frequency'] = freq

    # Concatenate all DataFrames in the dictionary
    mean_std_df = pd.concat(mean_std_dict.values(), ignore_index=True)

    # For each frequency after first one
    for i, freq in enumerate(subset_freqs[1:]):
        prev_freq = subset_freqs[i]  # Get previous frequency

        # For each row
        for idx, row in X.iterrows():
            sample = row['Sample']

            if HG_diff:
                # Get previous frequency's HG mean for this sample
                prev_hg = mean_std_df[
                    (mean_std_df['Frequency'] == prev_freq) &
                    (mean_std_df['Sample'] == sample)
                ]['HG_mean'].values[0]


                # 1) Inputs: xt - (xt-1) --First-order differences
                # 2) Inputs: (xt/(xt-1)) - 1 --Escalado relativo

                # Calculate and store difference
                X.loc[idx, f'{freq}.0 HG diff'] = X.loc[idx, f'{freq}.0 HG (mV) mean'] - prev_hg
                # X.loc[idx, f'{freq}.0 HG relative diff'] = (X.loc[idx, f'{freq}.0 HG (mV) mean'] / prev_hg) -1


            if LG_diff:
                prev_lg = mean_std_df[
                    (mean_std_df['Frequency'] == prev_freq) &
                    (mean_std_df['Sample'] == sample)
                ]['LG_mean'].values[0]

                # Calculate and store difference
                # X.loc[idx, f'{freq}.0 LG diff'] = X.loc[idx, f'{freq}.0 LG (mV) mean'] - prev_lg
                X.loc[idx, f'{freq}.0 LG relative diff'] = (X.loc[idx, f'{freq}.0 LG (mV) mean'] / prev_lg) -1


    # Drop the 'Sample' column
    X = X.drop(columns=['Sample'])

    return X


### Load New Test Data
Prepare new sample for testing (Testing other samples, out of initial dataset)

## Define training and testing data

In [8]:

# Subset of specific frequencies to use as input features, If freqs = [] then all frequencies are used

freqs = []

# Add a group with all frequencies from 100 to 590
# freqs.append(list(range(260, 591, 10)))
freqs.append(list(range(100, 591, 10)))


# Favourite frequencies
# freqs.append([250,300,320,330,340,350,360,370,380,390,400,410,420,430,440,450,460,470,480,490,500,510,520,530,540]) # TOP 25
# freqs.append([250,310,320,330,340,350,360,370,380,390,400,410,420,430,440,450,460,470,480,490,500,510,520,540]) # TOP 24

freqs.append([310,320,330,340,350,360,370,380,390,400,410,420,430,440,450,460,470,480,490,540]) # TOP 20 88% SG(3,2)
# freqs.append([310,320,330,340,350,360,370,400,410,420,430,440,450,460,470,480,490,500,510,540]) # TOP 20 88% SG(3,2)
# freqs.append([300,310,320,330,340,350,360,370,380,390,400,410,420,430,440,450,460,470,480,490]) # TOP 20 88% SG(3,2)

# freqs.append([310,320,330,340,350,360,370,380,390,400,410,420]) # TOP 12 (86%)
freqs.append([310,320,330,340,350,360,370,400,410,420]) # TOP 10 (88%)
freqs.append([330,340,350,360,410]) # TOP 5 86% SG

freqs.append([340,350,360]) # TOP 3
freqs.append([350]) # TOP 1


# # Count number of frequencies
# print(f'freqs number: {len(freqs)}')

# Array with subsets of frequencies to use for training
# freqs.extend([
#     [250, 300, 320],
#     [300, 310, 320],
#     [330, 340, 350],
#     [360, 370, 380],
#     [390, 400, 410],
#     [420, 430, 440]
# ])

# Add all frequencies one by one
# for i in range(250, 501, 10):
#     freqs.append([i])

# # Add groups of 2 frequencies
# for i in range(300, 411, 20):
#     if i + 10 <= 350:
#         freqs.append([i, i+10])

# # Add groups of 3 frequencies
# for i in range(250, 441, 30):
#     if i + 20 <= 590 and i + 10 <= 590:
#         freqs.append([i, i+10, i+20])



# freqs.append([250,320,330,410])

# freqs = freqs_fav ## TESTING

print(freqs)

# NOTE: legacy hardcoded subsets (paper uses the universal bands voted by the loop below).


[[100, 110, 120, 130, 140, 150, 160, 170, 180, 190, 200, 210, 220, 230, 240, 250, 260, 270, 280, 290, 300, 310, 320, 330, 340, 350, 360, 370, 380, 390, 400, 410, 420, 430, 440, 450, 460, 470, 480, 490, 500, 510, 520, 530, 540, 550, 560, 570, 580, 590], [310, 320, 330, 340, 350, 360, 370, 380, 390, 400, 410, 420, 430, 440, 450, 460, 470, 480, 490, 540], [310, 320, 330, 340, 350, 360, 370, 400, 410, 420], [330, 340, 350, 360, 410], [340, 350, 360], [350]]


## Testing

In [9]:
# LODO §1 — group tracking (NEW in train_lodo; train.ipynb frozen).
# Filenames encode the grouping key: A1_1.csv = polymer A, day 1; A3_25.csv = day 3, ... A2_13.csv = day 2, etc. Single directory holds all 5 days.
# LODO = 5 folds, one full day held out per fold.

def load_data_with_groups(input_path):
    """Like load_data_from_directory but keeps filename -> Day/SourceFile grouping keys."""
    frames = []
    for file in sorted(os.listdir(input_path)):
        if file.endswith('.csv'):
            df = pd.read_csv(os.path.join(input_path, file), delimiter=';', header=0)
            df['SourceFile'] = file
            # Day = leading number after polymer letter: A1_1 -> 1, A3_25 -> 3, G4_37 -> 4
            m = re.match(r'[A-Z](\d+)_', file)
            df['Day'] = int(m.group(1)) if m else -1
            frames.append(df)
    data = pd.concat(frames, ignore_index=True)
    data['Sample'] = data['Sample'].str[0]  # same cleaning as prepare_train_test_data
    return data

def load_grouped_data(notebook_dir,
                   train_dir='data/experiment_5_plastics/processed/'):
    """Load the single-directory long-format frame (all days; windowing happens AFTER, once)."""
    data = load_data_with_groups(os.path.normpath(os.path.join(notebook_dir, '..', '..', train_dir)))
    keep = [c for c in ['Frequency (GHz)', 'LG (mV)', 'HG (mV)', 'Sample', 'Day', 'SourceFile'] if c in data.columns]
    return data[keep]

In [10]:
# LODO §1 — windowing that carries Day/SourceFile (NEW in train_lodo).
# Grouping keys stay ['Sample', 'Frequency (GHz)'] IDENTICAL to baseline, so window
# boundaries and values match freq_as_variable exactly; each window additionally records
# the majority Day/SourceFile of its rows (only file-boundary windows mix days; ties
# broken deterministically by pandas mode = smallest value).

def _mode_first(s):
    m = s.mode()
    return m.iloc[0] if len(m) else s.iloc[0]

def grouped_window_averages(df, data_percentage):
    results = []
    for (sample, freq), group in df.groupby(['Sample', 'Frequency (GHz)']):
        window_size = max(1, int(len(group) * data_percentage / 100))
        for start in range(0, len(group), window_size):
            window_data = group.iloc[start:start + window_size]
            mean_values = window_data[['LG (mV)', 'HG (mV)']].mean()
            std_deviation_values = window_data[['LG (mV)', 'HG (mV)']].std()
            results.append({
                'Frequency (GHz)': freq,
                'LG (mV) mean': mean_values['LG (mV)'],
                'HG (mV) mean': mean_values['HG (mV)'],
                'LG (mV) std deviation': std_deviation_values['LG (mV)'],
                'HG (mV) std deviation': std_deviation_values['HG (mV)'],
                'Sample': sample,
                'Day': _mode_first(window_data['Day']),
                'SourceFile': _mode_first(window_data['SourceFile']),
            })
    return pd.DataFrame(results)

def grouped_pivot(df, data_percentage):
    """Same pivot as freq_as_variable + Day/SourceFile carried per (Sample, unique_id) row."""
    df_window = grouped_window_averages(df, data_percentage)
    df_window['unique_id'] = df_window.groupby(['Sample', 'Frequency (GHz)']).cumcount()
    feat = df_window.drop(columns=['Day', 'SourceFile'])
    df_pivot = feat.pivot(index=['Sample', 'unique_id'], columns='Frequency (GHz)')
    df_pivot.columns = [' '.join([str(col[1]), str(col[0])]) for col in df_pivot.columns]
    df_pivot = df_pivot.dropna(axis=1, how='all')
    df_pivot = df_pivot.reset_index()
    meta = (df_window.groupby(['Sample', 'unique_id'])
            .agg(Day=('Day', _mode_first), SourceFile=('SourceFile', _mode_first))
            .reset_index())
    df_pivot = df_pivot.merge(meta, on=['Sample', 'unique_id'], how='left')
    df_pivot = df_pivot.drop(columns=['unique_id'])
    feat_cols = sorted([c for c in df_pivot.columns if c not in ('Sample', 'Day', 'SourceFile')])
    return df_pivot[['Sample', 'Day', 'SourceFile'] + feat_cols]

In [11]:
# LODO §3 selection + §4 thickness/alpha defs (REPLACES hardcoded TOP lists for paper use).
# Spec note: thickness/alpha helpers live HERE (not Cell 34) so the Cell-32 LODO loop can
# use them top-to-bottom; Cell 34 only validates. train.ipynb keeps the old layout.

K_LIST = [50, 20, 10, 5, 3, 1]
FREQS_ALL = list(range(100, 591, 10))
MODELS_ORDER = ['RF', 'NB', 'LR', 'GB', 'SVM']


# ---- thickness per polymer letter, mm (same values as train.ipynb) ----
THICKNESS_MM = {
    'A': 0.20,   # PE/tie/EVOH/tie/PE/Adhesive/PE/tie/EVOH/tie/PE
    'B': 0.57,   # PE/tie/EVOH/tie/PE (Admer AT1707E)
    'C': 2.05,   # ABS+PC
    'D': 3.00,   # ABS
    'E': 0.10,   # Ecovio/PVOH/Ecovio
    'F': 0.29,   # PP/tie/EVOH/tie/PP (tupper, 0.27-0.31 -> 0.29)
    'G': 0.10,   # PHB/PVOH/Ecovio
    'H': 0.07,   # PP/tie/EVOH/tie/PP
    'I': 0.36,   # PS
    'J': 0.07,   # LDPE
    'L': 1.85,   # PVC
    'O': 0.12,   # PET
}

ALPHA_REF_FLOOR_MV = 0.5  # |HG median| below this ~= dead-band noise (per-window std ~= 0.13 mV)
ALPHA_LG_FLOOR_MV = 3.0  # LG twin (~2.5-3 sigma of per-window LG noise ~= 1.14 mV)

def compute_band_reference(df_pivot_train, freqs_all=FREQS_ALL, floor_mv=ALPHA_REF_FLOOR_MV, channel='HG'):
    """Per-frequency reference from TRAIN fold only: median of '{f}.0 {channel} (mV) mean'.

    No empty-system sweep exists for experiment 5, so the reference is the train-fold
    median per band. Computed inside each outer fold from TRAIN rows only (no leakage);
    the same ref transforms both the train and the held-out fold.
    Bands with non-finite, non-positive, or sub-floor |median| (HG: empirically 110-190 GHz
    plus weak 210/220/260/270/280; LG: 100 GHz plus ~290 GHz and everything 300+ GHz)
    get NaN: no signal lives there, only noise around zero.
    """
    ref = {}
    for f in freqs_all:
        col = f'{f}.0 {channel} (mV) mean'
        vals = pd.to_numeric(df_pivot_train[col], errors='coerce').dropna() if col in df_pivot_train.columns else pd.Series([], dtype=float)
        m = float(vals.median()) if len(vals) else float('nan')
        ref[f] = m if (np.isfinite(m) and m > 0 and abs(m) >= floor_mv) else float('nan')
    return ref

def apply_alpha_pivoted(df_pivot, ref_hg, ref_lg, freqs_all=FREQS_ALL, thickness_map=THICKNESS_MM, eps=1e-6):
    """Beer-Lambert alpha(f) = -ln(T)/d with T = sample(f)/ref(f), per channel.

    HG/LG means with live refs -> alpha; dead-band means -> exact 0 (anti-leak:
    a scaled constant would encode 1/d = the label, one thickness per polymer).
    All std-deviation cols pass through RAW (identical in both arms: std carries
    polymer signal with no systematic thickness scaling, so dividing it by d
    would only inject a 1/d code that exists solely in the alpha arm).
    Column names are preserved so preprocess_data/add_features work unchanged; frames
    stay separate per norm_mode. 'Sample'/'Day'/'SourceFile' columns unchanged.
    """
    out = df_pivot.copy()
    d = out['Sample'].map(thickness_map)
    if d.isna().any():
        raise ValueError(f"Missing thickness for samples: {out.loc[d.isna(), 'Sample'].unique()}")
    if bool((d <= 0).any()):
        raise ValueError("Non-positive thickness encountered")
    for f in freqs_all:
        hm = f'{f}.0 HG (mV) mean'
        if hm in out.columns:
            r = ref_hg.get(f, float('nan'))
            if r is not None and np.isfinite(r) and r > 0:
                T = pd.to_numeric(out[hm], errors='coerce') / r
                out[hm] = (-np.log(T.clip(lower=eps))) / d.values
            else:
                # Dead band: no signal, only noise. Constant 0 (NOT -ln(eps)/d):
                # a scaled constant would encode 1/d = the label (one thickness per
                # polymer) and let the alpha arm win via thickness, not spectroscopy.
                out[hm] = 0.0
        lm = f'{f}.0 LG (mV) mean'
        if lm in out.columns:
            r = ref_lg.get(f, float('nan'))
            if r is not None and np.isfinite(r) and r > 0:
                T = pd.to_numeric(out[lm], errors='coerce') / r
                out[lm] = (-np.log(T.clip(lower=eps))) / d.values
            else:
                # Dead LG band: same anti-leak rule as HG (constant 0, not 1/d).
                out[lm] = 0.0
    # NOTE: std-deviation cols intentionally untouched (raw in both arms).
    return out


def frequency_scores_from_importances(imp_by_model, feature_columns, freqs_all=FREQS_ALL):
    """Feature importances -> frequency scores.

    Each frequency owns up to 4 cols (HG/LG x mean/std). Per model: z-score its
    importances (scales differ across models), mean over the freq's present cols;
    then average across the 5 models. Returns a ranking table (best first).
    """
    z = {}
    for m, s in imp_by_model.items():
        s = pd.Series(np.asarray(s, dtype=float), index=list(feature_columns)).astype(float)
        sd = float(s.std(ddof=0))
        z[m] = (s - float(s.mean())) / (sd if sd > 0 else 1.0)
    rows = []
    for f in freqs_all:
        cols = [f'{f}.0 HG (mV) mean', f'{f}.0 LG (mV) mean',
                f'{f}.0 HG (mV) std deviation', f'{f}.0 LG (mV) std deviation']
        have = [c for c in cols if c in z['RF'].index]
        rows.append({'Frequency': f,
                     'score': float(np.mean([float(z[m][have].mean()) for m in z])) if have else float('nan'),
                     'n_cols': len(have)})
    tab = pd.DataFrame(rows).sort_values('score', ascending=False).reset_index(drop=True)
    tab['rank'] = tab.index + 1
    return tab

def select_topK_for_fold(Xtr_df, ytr, seed, K_list=K_LIST, freqs_all=FREQS_ALL):
    """Per-fold selection on TRAIN fold only (all 50 freqs) -> ({K: top-K}, ranking, models).

    Fits RF + LR + GB and permutation importance for NB/SVM via get_feature_importances
    (plot=False); Xtr_df must be a DataFrame (call BEFORE scaling/SG/PCA).
    """
    rf_m, nb_m, lr_m, gb_m, svm_m, _ = train_models(Xtr_df, ytr, seed)
    # NOTE: the shared helper scores LR with coef_[0] (first class only in multinomial).
    # Universal selection uses mean |coef| over classes (strictly more complete); the
    # helper is still called for the RF/GB/NB/SVM importances.
    rf_imp, _lr_imp_legacy, gb_imp, nb_imp, svm_imp = get_feature_importances(
        rf_m, lr_m, gb_m, nb_m, svm_m, Xtr_df, ytr, seed, False, 10)
    lr_full = pd.DataFrame({'Feature': [str(c) for c in Xtr_df.columns],
                            'Importance': np.asarray(np.abs(_lr_coef(lr_m)).mean(axis=0), dtype=float)})
    imp = {}
    for name, dfi in [('RF', rf_imp), ('LR', lr_full), ('GB', gb_imp), ('NB', nb_imp), ('SVM', svm_imp)]:
        imp[name] = pd.Series(np.asarray(dfi['Importance'], dtype=float),
                              index=[str(c) for c in dfi['Feature']])
    feat_cols = [str(c) for c in Xtr_df.columns]
    ranking = frequency_scores_from_importances(imp, feat_cols, freqs_all)
    topK = {int(K): ranking['Frequency'].head(int(K)).tolist() for K in K_list}
    return topK, ranking, (rf_m, nb_m, lr_m, gb_m, svm_m)


In [ ]:
OUT_PREFIX = 'lodo_copy_'  # separate outputs; same workflow as train_lodo (reproduces its numbers)

# LODO main loop (REPLACES the row-split testing loop; no train_test_split here).
# Guards: reference + selection + scaler/SG/PCA are fit on the TRAIN fold only and applied
# to the held-out fold; the held-out day is never touched except for final scoring.
# Per-fold rows are saved raw; aggregation to mean±std happens in the stability cell below.

SMOKE = False  # True -> 1 fold, K=[10, 3]
WINDOW_S = 0.1
LABELS = ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'L', 'O']
NORM_MODES = ['baseline', 'alpha']  # paper loop (deprecated x/d dropped: -33% fits)
K_VALUES = list(K_LIST) if not SMOKE else [10, 3]
# transform flags mirror the train.ipynb comparison flags (fit TRAIN-only, apply to test)
APPLY_SCALING = False
APPLY_SG = False
SG_W = 3
SG_P = 2
APPLY_PCA = False
APPLY_LDA = False
APPLY_QDA = False
APPLY_ICA = False
# K=50 model reuse below is valid only while every transform stays off (raw path)
_raw_path = not (APPLY_SCALING or APPLY_SG or APPLY_PCA or APPLY_LDA or APPLY_QDA or APPLY_ICA)

WINDOW_PCT = (100 / 12) * WINDOW_S  # single percentage: windowing BEFORE the split
# pivot cache: windowing is deterministic; re-runs reuse it when inputs + window match.
# NOTE: pandas' own to_parquet/read_parquet is deliberately NOT used: its process-global
# extension-type hotfix has failed twice in long-lived kernels (ArrowKeyError at
# register/unregister). Direct pyarrow calls register nothing and are immune; pickle
# is the fallback if Arrow itself fails. Format recorded in the sidecar.
_pivot_cache = os.path.normpath(os.path.join(notebook_dir, '..', '..', 'results/exp_5/lodo_pivot_cache'))
_pivot_meta = _pivot_cache + '.json'
def _pivot_cache_write(df):
    try:
        import pyarrow as _pa
        import pyarrow.parquet as _pq
        _pq.write_table(_pa.Table.from_pandas(df, preserve_index=False), _pivot_cache + '.parquet')
        return 'parquet'
    except Exception as _e:
        print(f'parquet cache failed ({type(_e).__name__}: {_e}); falling back to pickle')
        df.to_pickle(_pivot_cache + '.pkl')
        return 'pickle'
def _pivot_cache_read(fmt):
    if fmt == 'parquet':
        import pyarrow.parquet as _pq
        return _pq.read_table(_pivot_cache + '.parquet').to_pandas()
    return pd.read_pickle(_pivot_cache + '.pkl')
def _pivot_input_sig():
    h = _hl.sha256()
    _d = os.path.normpath(os.path.join(notebook_dir, '..', '..', 'data/experiment_5_plastics/processed'))
    for _f in sorted(os.listdir(_d)):
        if _f.endswith('.csv'):
            _st = os.stat(os.path.join(_d, _f))
            h.update(f'{_f}:{_st.st_size}:{int(_st.st_mtime)}'.encode())
    try:
        import pyarrow as _pa_sig
        _arrow_v = _pa_sig.__version__
    except Exception:
        _arrow_v = None
    return {'window_s': WINDOW_S, 'dp': WINDOW_PCT, 'inputs': h.hexdigest(),
            'pandas': pd.__version__, 'pyarrow': _arrow_v}
_sig = _pivot_input_sig()
df_pivot_full = None
if os.path.exists(_pivot_meta):
    try:
        _meta = json.load(open(_pivot_meta, encoding='utf-8'))
        if _meta.get('sig') == _sig:
            df_pivot_full = _pivot_cache_read(_meta.get('fmt', 'parquet'))
            print(f'LODO pivot: loaded cache [{_meta.get("fmt")}] {df_pivot_full.shape}')
    except Exception as _e:
        print(f'pivot cache unreadable ({type(_e).__name__}: {_e}); rebuilding')
        try:
            import pandas as _pd_probe, pyarrow as _pa_probe
            from pandas.compat import pa_version_under14p1 as _old
            print(f'env probe: pandas {_pd_probe.__version__} @ {_pd_probe.__file__}')
            print(f'env probe: pyarrow {_pa_probe.__version__} @ {_pa_probe.__file__}')
            print(f'env probe: pa_version_under14p1={_old}; sys.path={_sys.path[:5]}')
        except Exception as _e2:
            print(f'env probe failed: {_e2}')
        df_pivot_full = None
if df_pivot_full is None:
    _df_long = load_grouped_data(notebook_dir)
    df_pivot_full = grouped_pivot(_df_long, WINDOW_PCT).dropna().reset_index(drop=True)
    _fmt = _pivot_cache_write(df_pivot_full)
    json.dump({'sig': _sig, 'fmt': _fmt}, open(_pivot_meta, 'w', encoding='utf-8'), indent=2, sort_keys=True)
    print(f'LODO pivot: built + cached [{_fmt}] {df_pivot_full.shape}')
_days = sorted(df_pivot_full['Day'].unique())
print(f'LODO pivot: {df_pivot_full.shape}, days={_days}')
assert set(_days) == {1, 2, 3, 4, 5}, f'expected days 1..5, got {_days}'
_groups = df_pivot_full['Day'].values
_gkf = GroupKFold(n_splits=5)

fold_records, topk_by_fold, rankings_by_fold, band_refs, band_refs_lg = [], {}, {}, {}, {}
_df_test_viz = None
for _fold, (_tri, _tei) in enumerate(_gkf.split(df_pivot_full, groups=_groups)):
    if SMOKE and _fold > 0:
        break
    _held = sorted(df_pivot_full.iloc[_tei]['Day'].unique())
    assert len(_held) == 1, f'fold {_fold} mixes days: {_held}'
    _held = _held[0]
    print(f'--- fold {_fold}: held-out day {_held} (n_train={len(_tri)}, n_test={len(_tei)}) ---')
    _df_tr = df_pivot_full.iloc[_tri].reset_index(drop=True)
    _df_te = df_pivot_full.iloc[_tei].reset_index(drop=True)
    if _fold == 0:
        _df_test_viz = _df_te
    for _norm in NORM_MODES:
        if _norm == 'baseline':
            _tr_n, _te_n, _ref, _ref_lg = _df_tr.copy(), _df_te.copy(), None, None
        elif _norm == 'alpha':
            _ref = compute_band_reference(_df_tr, FREQS_ALL)  # TRAIN fold only
            _ref_lg = compute_band_reference(_df_tr, FREQS_ALL, ALPHA_LG_FLOOR_MV, 'LG')
            _tr_n = apply_alpha_pivoted(_df_tr, _ref, _ref_lg, FREQS_ALL)
            _te_n = apply_alpha_pivoted(_df_te, _ref, _ref_lg, FREQS_ALL)
        band_refs[(_fold, _norm)] = _ref
        band_refs_lg[(_fold, _norm)] = _ref_lg
        _Xtr50, _ytr50 = preprocess_data(_tr_n, LABELS, FREQS_ALL)
        _Xtr50 = add_features(_Xtr50, _ytr50, FREQS_ALL, False, False)
        _topK, _ranking, _sel_models = select_topK_for_fold(_Xtr50, _ytr50, seed)
        topk_by_fold[(_fold, _norm)] = _topK
        rankings_by_fold[(_fold, _norm)] = _ranking
        for _K in K_VALUES:
            _fq = _topK[int(_K)]
            _Xtr, _ytr = preprocess_data(_tr_n, LABELS, _fq)
            _Xtr = add_features(_Xtr, _ytr, _fq, False, False)
            _Xte, _yte = preprocess_data(_te_n, LABELS, _fq)
            _Xte = add_features(_Xte, _yte, _fq, False, False)
            if APPLY_SCALING:
                _sc = StandardScaler()
                _Xtr = _sc.fit_transform(_Xtr)
                _Xte = _sc.transform(_Xte)
            if APPLY_SG:
                _Xtr = savgol_filter(_Xtr, window_length=SG_W, polyorder=SG_P)
                _Xte = savgol_filter(_Xte, window_length=SG_W, polyorder=SG_P)
            if APPLY_PCA:
                _pc = PCA(n_components=0.95, random_state=seed)
                _Xtr = _pc.fit_transform(_Xtr)
                _Xte = _pc.transform(_Xte)
            if APPLY_LDA:
                _ld = LinearDiscriminantAnalysis()
                _Xtr = _ld.fit_transform(_Xtr, _ytr)
                _Xte = _ld.transform(_Xte)
            if APPLY_QDA:
                _qd = QuadraticDiscriminantAnalysis()
                _qd.fit(_Xtr, _ytr)
                _Xtr = np.hstack((_Xtr, _qd.predict_proba(_Xtr)))
                _Xte = np.hstack((_Xte, _qd.predict_proba(_Xte)))
            if APPLY_ICA:
                _ic = FastICA(n_components=2, random_state=seed)
                _Xtr = _ic.fit_transform(_Xtr)
                _Xte = _ic.transform(_Xte)
            _n_feat = _Xtr.shape[1]
            if int(_K) == 50 and _raw_path:
                _models = list(_sel_models)  # same matrix + seed as selection fit: skip refit
            else:
                _models = train_models(_Xtr, _ytr, seed)
            for _mi, _mn in enumerate(MODELS_ORDER):
                _yp = _models[_mi].predict(_Xte)
                fold_records.append({
                    'fold': _fold, 'held_out_day': _held, 'norm_mode': _norm,
                    'K': int(_K), 'model': _mn,
                    'acc': float(_acc(_yte, _yp)),
                    'prec': float(_prec(_yte, _yp, average='weighted', zero_division=0)),
                    'rec': float(_rec(_yte, _yp, average='weighted', zero_division=0)),
                    'f1': float(_f1(_yte, _yp, average='weighted', zero_division=0)),
                    'n_feat': int(_n_feat),
                    'selected_freqs': ','.join(str(f) for f in _fq),
                })

cv_results = pd.DataFrame(fold_records)
_out_csv = os.path.normpath(os.path.join(notebook_dir, '..', '..', 'results/exp_5', OUT_PREFIX + 'per_fold.csv'))
os.makedirs(os.path.dirname(_out_csv), exist_ok=True)
cv_results.to_csv(_out_csv, index=False, sep=';')
print(f'Saved per-fold results -> {_out_csv} ({cv_results.shape[0]} rows)')

# viz-compat globals for the PCA/3D cells below (full pivoted frame left behind)
labels = list(LABELS)
eliminate_std_dev = False
eliminate_LG = False
HG_diff = False
LG_diff = False
apply_scaling = False
apply_savitzky_golay = False
apply_pca = False
apply_lda = False
apply_qda = False
apply_ica = False
df_train = df_pivot_full
df_test = _df_test_viz
subset_freqs = list(FREQS_ALL)

LODO pivot: loaded cache [parquet] (1445, 203)
LODO pivot: (1445, 203), days=[np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5)]
--- fold 0: held-out day 4 (n_train=1152, n_test=293) ---
--- fold 1: held-out day 2 (n_train=1152, n_test=293) ---
--- fold 2: held-out day 5 (n_train=1156, n_test=289) ---
--- fold 3: held-out day 1 (n_train=1159, n_test=286) ---


In [ ]:
# LODO guards (post-loop diagnostics; read-only, no refit of paper numbers).
_n_folds = cv_results['fold'].nunique()
print(f'folds={_n_folds}, norms={sorted(cv_results["norm_mode"].unique())}, K={sorted(cv_results["K"].unique())}')
# 1. fold coverage: one held-out day per fold, no repeats
_cov = cv_results[['fold', 'held_out_day']].drop_duplicates().sort_values('fold')
print(_cov.to_string(index=False))
assert _cov['held_out_day'].is_unique, 'a day was held out twice'
if not SMOKE:
    assert set(_cov['held_out_day']) == {1, 2, 3, 4, 5}, 'LODO must cover days 1..5'
# 2. train/test days disjoint per fold (GroupKFold guarantee, verified on the frame)
_gkf_check = GroupKFold(n_splits=5)
for _f, (_a, _b) in enumerate(_gkf_check.split(df_pivot_full, groups=df_pivot_full['Day'].values)):
    if SMOKE and _f > 0:
        break
    assert set(df_pivot_full.iloc[_a]['Day']) & set(df_pivot_full.iloc[_b]['Day']) == set()
print('day overlap check: OK (train/test days disjoint per fold)')
# 3. Day balance of the pivoted frame
print(df_pivot_full.groupby('Day').size().to_string())
# 4. reference sanity: live refs finite+positive; dead bands NaN (train-fold only).
#    Core dead set (full-data |median| < 0.02 << floor) must be degenerate in every
#    fold; the full per-fold dead set is reported, not asserted (marginals may flip).
_core_dead = {110, 120, 130, 140, 150, 160, 170, 180, 190}
for (_f, _n), _r in sorted(band_refs.items()):
    if _n != 'alpha' or (SMOKE and _f > 0):
        continue
    _live = {f for f in FREQS_ALL if np.isfinite(_r.get(f, float('nan'))) and _r[f] > 0}
    _dead = set(FREQS_ALL) - _live
    assert _core_dead <= _dead, f'fold {_f}: core dead bands live?! {sorted(_core_dead - _dead)}'
    print(f'fold {_f}: {len(_live)} live / {len(_dead)} dead bands; dead={sorted(_dead)}')
print('reference check: OK (train-fold medians; dead = NaN, never scaled)')
# 4b. LG reference sanity (mirror of HG: core LG-dead set asserted, marginals reported).
_core_dead_lg = {100} | set(range(310, 591, 10))
for (_f, _n), _r in sorted(band_refs_lg.items()):
    if _n != 'alpha' or (SMOKE and _f > 0):
        continue
    _live = {f for f in FREQS_ALL if np.isfinite(_r.get(f, float('nan'))) and _r[f] > 0}
    _dead = set(FREQS_ALL) - _live
    assert _core_dead_lg <= _dead, f'fold {_f}: core LG-dead bands live?! {sorted(_core_dead_lg - _dead)}'
    print(f'fold {_f} LG: {len(_live)} live / {len(_dead)} dead bands; dead={sorted(_dead)}')
print('LG reference check: OK (train-fold medians; dead = NaN, never scaled)')
# 5. alpha finiteness + no-leak check on the fold-0 held-out frame (same transform as loop)
_r0 = band_refs[(0, 'alpha')]
_r0_lg = band_refs_lg[(0, 'alpha')]
_day0 = int(_cov.iloc[0]['held_out_day'])
_a0 = apply_alpha_pivoted(
    df_pivot_full[df_pivot_full['Day'] == _day0].reset_index(drop=True), _r0, _r0_lg, FREQS_ALL)
_num0 = _a0.drop(columns=['Sample', 'Day', 'SourceFile']).apply(pd.to_numeric, errors='coerce')
assert bool(np.isfinite(_num0.values).all()), 'non-finite alpha values'
_dead0 = [f for f in FREQS_ALL if not (np.isfinite(_r0.get(f, float('nan'))) and _r0[f] > 0)]
for _f in _dead0:
    _col = _a0[f'{_f}.0 HG (mV) mean'].values
    assert bool((_col == 0.0).all()), f'dead band {_f} not zero-filled (thickness leak!)'
_dead0_lg = [f for f in FREQS_ALL if not (np.isfinite(_r0_lg.get(f, float('nan'))) and _r0_lg[f] > 0)]
for _f in _dead0_lg:
    _col = _a0[f'{_f}.0 LG (mV) mean'].values
    assert bool((_col == 0.0).all()), f'dead LG band {_f} not zero-filled (thickness leak!)'
print(f'alpha checks OK: finite ({_num0.shape[1]} cols), {len(_dead0)} dead HG + {len(_dead0_lg)} dead LG bands exactly 0 (no thickness leak)')
_raw0 = df_pivot_full[df_pivot_full['Day'] == _day0].reset_index(drop=True)
for _c in [c for c in _a0.columns if 'std deviation' in c]:
    assert bool((pd.to_numeric(_a0[_c], errors='coerce').values
                 == pd.to_numeric(_raw0[_c], errors='coerce').values).all()), \
        f'std col {_c} modified by alpha transform (must stay raw in both arms)'
print('std check OK: std-deviation cols pass through raw (symmetric arms)')
# 6. reference sensitivity: train-fold median ref vs day-1-only ref (vectors only, no eval)
for (_f, _n), _r in sorted(band_refs.items()):
    if _n != 'alpha' or (SMOKE and _f > 0):
        continue
    _r1 = compute_band_reference(df_pivot_full[df_pivot_full['Day'] == 1], FREQS_ALL)
    _a = np.array([_r[f] for f in FREQS_ALL], dtype=float)
    _b = np.array([_r1[f] for f in FREQS_ALL], dtype=float)
    _rel = np.abs(_a - _b) / np.maximum(np.abs(_b), 1e-12)
    print(f'fold {_f}: max |ref_train - ref_day1|/ref_day1 = {_rel.max():.3f} (median {np.median(_rel):.3f})')


In [ ]:
# Stability table + mean±std summary (exports the CSVs the manuscript must match).
_paper_norms = ['baseline', 'alpha']  # deprecated x/d excluded from paper tables
_n_folds_eff = cv_results['fold'].nunique()
_req = 4 if _n_folds_eff == 5 else _n_folds_eff  # universal threshold
print(f'universal threshold: >= {_req}/{_n_folds_eff} folds')

_stab_rows = []
for _n in _paper_norms:
    for _K in sorted(cv_results['K'].unique()):
        _cnt, _ranks = Counter(), {}
        for _f in sorted(cv_results['fold'].unique()):
            for _freq in topk_by_fold[(_f, _n)][int(_K)]:
                _cnt[_freq] += 1
            for _freq, _r in rankings_by_fold[(_f, _n)].set_index('Frequency')['rank'].items():
                _ranks.setdefault(_freq, []).append(int(_r))
        for _freq in FREQS_ALL:
            _sel = int(_cnt.get(_freq, 0))
            _rr = _ranks.get(_freq, [])
            _stab_rows.append({'norm_mode': _n, 'K': int(_K), 'freq': int(_freq),
                               'folds_selected': f'{_sel}/{_n_folds_eff}',
                               'n_selected': _sel,
                               'mean_rank': round(float(np.mean(_rr)), 2) if _rr else None,
                               'in_universal': bool(_sel >= _req)})
stability_df = pd.DataFrame(_stab_rows)
_sp = os.path.normpath(os.path.join(notebook_dir, '..', '..', 'results/exp_5', OUT_PREFIX + 'stability.csv'))
stability_df.to_csv(_sp, index=False, sep=';')

universal = {}
for _n in _paper_norms:
    for _K in sorted(cv_results['K'].unique()):
        _sub = stability_df[(stability_df['norm_mode'] == _n) & (stability_df['K'] == int(_K)) & stability_df['in_universal']]
        universal[(_n, int(_K))] = sorted(_sub['freq'].tolist())
        print(f'universal-{_K} [{_n}]: {universal[(_n, int(_K))]}')

summary_df = (cv_results[cv_results['norm_mode'].isin(_paper_norms)]
            .groupby(['norm_mode', 'K', 'model'])
            .agg(mean_acc=('acc', 'mean'), std_acc=('acc', 'std'),
                 mean_f1=('f1', 'mean'), std_f1=('f1', 'std'), n_folds=('fold', 'nunique'))
            .reset_index().round(4))
_su = os.path.normpath(os.path.join(notebook_dir, '..', '..', 'results/exp_5/lodosummary_df.csv'))
summary_df.to_csv(_su, index=False, sep=';')
print(f'Saved {_sp} ({stability_df.shape[0]} rows) and {_su} ({summary_df.shape[0]} rows)')
display(summary_df.pivot_table(index=['K', 'model'], columns='norm_mode', values='mean_acc').round(4))

In [ ]:
# LODO plots: mean±std baseline vs alpha per K + selection-stability bars,
# with percentages annotated.

_outdir = os.path.normpath(os.path.join(notebook_dir, '..', '..', 'results/exp_5/'))
os.makedirs(_outdir, exist_ok=True)

_x = np.arange(len(MODELS_ORDER))
_w = 0.35

for _K in sorted(cv_results['K'].unique()):
    _fig, _ax = plt.subplots(figsize=(9, 4.5))

    for _j, _n in enumerate(['baseline', 'alpha']):
        _mu, _sd = [], []

        for _m in MODELS_ORDER:
            _r = cv_results[
                (cv_results['norm_mode'] == _n) &
                (cv_results['K'] == int(_K)) &
                (cv_results['model'] == _m)
            ]['acc']

            _mu.append(float(_r.mean()))
            _sd.append(float(_r.std(ddof=1)) if len(_r) > 1 else 0.0)

        _bars = _ax.bar(
            _x + (_j - 0.5) * _w,
            _mu,
            _w,
            yerr=_sd,
            capsize=3,
            label=('baseline T(f)' if _n == 'baseline'
                   else 'alpha(f) = -ln(T)/d')
        )

        for _bar, _value in zip(_bars, _mu):
            _ax.text(
                _bar.get_x() + _bar.get_width() / 2,
                _bar.get_height() + 0.02,
                f'{_value:.1%}',
                ha='center',
                va='bottom',
                fontsize=8
            )

    _ax.set_xticks(_x, MODELS_ORDER)
    _ax.set_ylim(0, 1.12)
    _ax.set_ylabel('Accuracy (mean±std over LODO folds)')
    _ax.set_title(f'LODO held-out-day accuracy | K={_K}')
    _ax.legend()
    _ax.grid(True, axis='y', alpha=0.3)
    plt.tight_layout()
    plt.savefig(
        os.path.join(_outdir, f'{OUT_PREFIX}compare_{_K}freqs.pdf'),
        bbox_inches='tight',
        dpi=300
    )
    plt.close()

for _K in [k for k in (20, 10, 5, 3)
           if k in set(int(k) for k in cv_results['K'].unique())]:

    _fig, _ax = plt.subplots(figsize=(10, 4))
    _fr = list(FREQS_ALL)
    _xi = np.arange(len(_fr))
    _w2 = 0.35
    _n_folds = cv_results['fold'].nunique()

    for _j, _n in enumerate(['baseline', 'alpha']):
        _s = stability_df[
            (stability_df['norm_mode'] == _n) &
            (stability_df['K'] == int(_K))
        ].set_index('freq')

        _vals = [
            int(_s.loc[f, 'n_selected']) if f in _s.index else 0
            for f in _fr
        ]

        _bars = _ax.bar(
            _xi + (_j - 0.5) * _w2,
            _vals,
            _w2,
            label=_n
        )

        for _bar, _value in zip(_bars, _vals):
            _percentage = 100 * _value / _n_folds
            _ax.text(
                _bar.get_x() + _bar.get_width() / 2,
                _value + 0.05,
                f'{_percentage:.0f}%',
                ha='center',
                va='bottom',
                fontsize=6,
                rotation=90
            )

    _thr = 4 if _n_folds == 5 else _n_folds
    _ax.axhline(_thr, color='k', ls='--', lw=1,
                label='universal threshold')
    _ax.set_xticks(_xi, _fr, rotation=90, fontsize=7)
    _ax.set_ylabel('Folds selected (%)')
    _ax.set_ylim(0, _n_folds + 1.2)
    _ax.set_title(
        f'Selection stability K={_K} '
        f'(universal ≥{_thr}/{_n_folds} folds)'
    )
    _ax.legend()
    _ax.grid(True, axis='y', alpha=0.3)
    plt.tight_layout()
    plt.savefig(
        os.path.join(_outdir, f'{OUT_PREFIX}stability_K{_K}.pdf'),
        bbox_inches='tight',
        dpi=300
    )
    plt.close()

In [ ]:
# Fold-0 confusion matrices: best K/model per norm; refit = deterministic replay of the loop fit.
_cm_dir = os.path.normpath(os.path.join(notebook_dir, '..', '..', 'results/exp_5'))
os.makedirs(_cm_dir, exist_ok=True)
_held0 = int(cv_results[cv_results['fold'] == 0]['held_out_day'].iloc[0])
_df_tr0 = df_pivot_full[df_pivot_full['Day'] != _held0].reset_index(drop=True)
_df_te0 = df_pivot_full[df_pivot_full['Day'] == _held0].reset_index(drop=True)
_fitters = {
    'RF': lambda: RandomForestClassifier(random_state=seed),
    'NB': lambda: GaussianNB(),
    'LR': lambda: make_pipeline(StandardScaler(), LogisticRegression(random_state=seed, max_iter=5000)),
    'GB': lambda: GradientBoostingClassifier(random_state=seed),
    'SVM': lambda: SVC(random_state=seed),
}
for _norm in NORM_MODES:
    _sub = cv_results[(cv_results['fold'] == 0) & (cv_results['norm_mode'] == _norm)]
    _best = _sub.loc[_sub['acc'].idxmax()]
    _K, _model = int(_best['K']), str(_best['model'])
    _fq = [int(f) for f in str(_best['selected_freqs']).split(',')]
    if _norm == 'baseline':
        _tr_n, _te_n = _df_tr0.copy(), _df_te0.copy()
    else:
        _tr_n = apply_alpha_pivoted(_df_tr0, band_refs[(0, _norm)], band_refs_lg[(0, _norm)], FREQS_ALL)
        _te_n = apply_alpha_pivoted(_df_te0, band_refs[(0, _norm)], band_refs_lg[(0, _norm)], FREQS_ALL)
    _Xtr, _ytr = preprocess_data(_tr_n, LABELS, _fq)
    _Xtr = add_features(_Xtr, _ytr, _fq, False, False)
    _Xte, _yte = preprocess_data(_te_n, LABELS, _fq)
    _Xte = add_features(_Xte, _yte, _fq, False, False)
    _cm_model = _fitters[_model]()
    _cm_model.fit(_Xtr, _ytr)
    _yp = _cm_model.predict(_Xte)
    print(f'{_norm}: K={_K} model={_model} acc={float((_yp == _yte.values).mean()):.4f}')
    plot_confusion_matrix_pdf(_yte, _yp, LABELS, _cm_dir, f'{OUT_PREFIX}fold0_{_norm}_{_model}_K{_K}')


### Principal Component Analysis

In [ ]:
def adjust_color_brightness(color, factor=0.6):
    return tuple([min(1, max(0, c * factor)) for c in color])



# PCA 3D Visualization
# Generate multiple views of the same PCA result
for freq in freqs:
    subset_freqs = freq
    print(f'Frequency: {freq}')

    X_train, y_train = preprocess_data(df_train, labels, subset_freqs, eliminate_std_dev, eliminate_LG, drop_sample=True)
    X_test, y_test = preprocess_data(df_test, labels, subset_freqs, eliminate_std_dev, eliminate_LG, drop_sample=True)

    X_train = add_features(X_train, y_train, subset_freqs, HG_diff, LG_diff)
    X_test = add_features(X_test, y_test, subset_freqs, HG_diff, LG_diff)

    pca = PCA(n_components=0.95, random_state=seed)
    X_train_pca = pca.fit_transform(X_train)
    # Print all PCA component percentages
    print("Explained variance by PCA components:")
    for i, var in enumerate(pca.explained_variance_ratio_):
        print(f"PC{i+1}: {var*100:.2f}%")
    print(f"Total variance explained: {sum(pca.explained_variance_ratio_)*100:.2f}%")
    print(f"Number of components: {len(pca.explained_variance_ratio_)}")
    X_test_pca = pca.transform(X_test)
    y_pca = y_train

    # # Generate different views of the same data
    # views = [
    #     (30, -45),     # Default view
    #     # (0, 0),      # Front view
    #     # (0, 90),     # Side view
    #     # (90, 0),     # Top view
    #     (20, -70),
    #     (30, -20),
    #     (25, -25),
    #     (35, -75),

    # ]

    # for elev, azim in views:
    #     plot_pca_3d(X_train_pca, y_pca, elev=elev, azim=azim, show_labels=False)


### 3D Visualization (Frequencies)

In [ ]:
def adjust_color_brightness(color, factor=0.6):
    return tuple([min(1, max(0, c * factor)) for c in color])

def plot_3d_specific_frequencies(df, freq_names, elev=30, azim=-45):
    """
    Plot 3D visualization using 3 selected frequencies directly from the dataframe.

    Parameters:
    -----------
    df : DataFrame
        DataFrame containing the frequency data
    freq_names : list
        List of 3 column names to visualize (e.g., ['340.0 HG (mV)', '350.0 HG (mV)', '360.0 HG (mV)'])
    elev : float, default=30
        Elevation angle for 3D plot viewing
    azim : float, default=-45
        Azimuth angle for 3D plot viewing
    """
    # Check if we have 3 frequencies
    if len(freq_names) != 3:
        raise ValueError("Need exactly 3 frequency names for 3D plot")

    # Check if frequencies exist in dataframe
    for freq in freq_names:
        if freq not in df.columns:
            raise ValueError(f"Frequency column '{freq}' not found in dataframe")

    # Extract data for the 3 frequencies and the sample labels
    X_3d = df[freq_names].values
    y = df['Sample'].values

    # Encode labels
    le = LabelEncoder()
    y_encoded = le.fit_transform(y)
    unique_classes = np.unique(y_encoded)
    n_classes = len(unique_classes)

    # Define color palette using Seaborn's husl palette
    base_palette = sns.color_palette("husl", n_classes)

    # Create a mapping of labels to colors
    color_dict = {}
    for i, label in enumerate(le.classes_):
        color_dict[label] = base_palette[i]

    # Create figure
    fig = plt.figure(figsize=(10, 8))
    ax = fig.add_subplot(111, projection='3d')

    # Set the viewing angle
    ax.view_init(elev=elev, azim=azim)

    # Plot each class separately for better legend
    for i, class_name in enumerate(le.classes_):
        mask = y == class_name
        color = base_palette[i]
        ax.scatter(X_3d[mask, 0],
                  X_3d[mask, 1],
                  X_3d[mask, 2],
                  color=color,
                  edgecolor=adjust_color_brightness(color, 0.6),
                  label=class_name,
                  s=25,  # dot size
                  alpha=0.6)

    # Extract frequency values from column names for axis labels
    freq_values = []
    for name in freq_names:
        # Extract the numeric part from strings like "340.0 HG (mV)"
        match = re.search(r'(\d+\.?\d*)', name)
        if match:
            freq_values.append(match.group(1))
        else:
            freq_values.append(name)

    # Set axis labels with frequency values
    ax.set_xlabel(f'{freq_names[0]}')
    ax.set_ylabel(f'{freq_names[1]}')
    ax.set_zlabel(f'{freq_names[2]}')

    # Add grid for better depth perception
    ax.grid(True)

    # Add legend with class names
    plt.legend(loc='best', markerscale=1.5)

    # Set title
    plt.title(f'3D Visualization of Selected Frequencies\n{freq_values[0]}, {freq_values[1]}, {freq_values[2]} GHz')

    # Save figure with angle and frequency information in filename
    filepath = os.path.normpath(os.path.join(notebook_dir, '..', '..', 'results/freq_viz/'))
    if not os.path.exists(filepath):
        os.makedirs(filepath)

    # Create a cleaner filename
    clean_freqs = [re.sub(r'[^\d.]', '', f) for f in freq_values]
    freqs_code = '_'.join(clean_freqs)
    plt.savefig(f'{filepath}/freq_viz_{freqs_code}_elev{elev}_azim{azim}.pdf', bbox_inches='tight', dpi=300)
    plt.show()

    return ax


# Define specific target frequencies to visualize
target_freqs = [340.0, 350.0, 360.0]  # Adjust to your desired frequencies

# Find matching columns in the original dataframe
hg_columns = []
for target_freq in target_freqs:
    # Find columns that match this frequency
    for col in df_train.columns:
        if f"{target_freq} HG (mV)" in col:
            hg_columns.append(col)
            break

if len(hg_columns) == 3:
    # Use the original dataframe with the identified columns
    df_viz = df_train.copy()

    # Apply preprocessing to these specific columns
    if apply_savitzky_golay:
        for col in hg_columns:
            df_viz[col] = savgol_filter(df_viz[col].values,
                                      window_length=5, polyorder=3)

    # Generate different viewing angles
    views = [
        (30, -45),   # Default view
        (20, -70),
        (30, -20),
        (10, -120),
    ]

    for elev, azim in views:
        plot_3d_specific_frequencies(df_viz, hg_columns, elev=elev, azim=azim)
else:
    print(f"Could not find all required frequency columns. Found: {hg_columns}")
    print(f"Available columns: {[col for col in df_train.columns if 'HG (mV)' in col]}")

In [ ]:
subset_freqs = list(range(100, 600, 10))

X_train, y_train = preprocess_data(df_train, labels, subset_freqs, eliminate_std_dev, eliminate_LG, drop_sample=True)
X_train = add_features(X_train, y_train, subset_freqs, HG_diff, LG_diff)

## NON PCA VISUALIZATION ##
# Choose specific variables for visualization
var1 = '410.0 HG (mV)'
var2 = '360.0 HG (mV)'

try:
    X_train[var1].describe()
except Exception as e:
    var1 = f'{var1} mean'
    var2 = f'{var2} mean'

def plot_data_visualization(X_train, y_train, var1, var2):

    # Get unique classes and encode
    le = LabelEncoder()
    y_encoded = le.fit_transform(y_train)
    n_classes = len(np.unique(y_encoded))

    # Create custom colormap with only needed colors
    colors = plt.cm.tab20(np.linspace(0, 1, 20))  # Get all 20 colors
    colors = colors[:n_classes]  # Take only needed colors
    custom_cmap = plt.cm.colors.ListedColormap(colors)

    # Create plot
    plt.figure(figsize=(10, 8))
    scatter = plt.scatter(X_train[var1],
                         X_train[var2],
                         c=y_encoded,
                         cmap=custom_cmap,
                         edgecolor='k',
                         s=25)

    # Create custom legend
    legend_elements = [plt.Line2D([0], [0],
                                marker='o',
                                color='w',
                                markerfacecolor=colors[i],
                                label=class_name,
                                markersize=10)
                      for i, class_name in enumerate(le.classes_)]

    plt.legend(handles=legend_elements, title="Classes")
    plt.xlabel(var1)
    plt.ylabel(var2)
    plt.show()

plot_data_visualization(X_train, y_train, var1, var2)